# Lab 0 — Getting Started

**ECON 282E · Foundations of Macroeconomics · UC Riverside, Fall 2026**

Do this **before Session 3 (October 8)**. It installs the tools, checks that they work, and
writes your first two programs. Budget 45 minutes, most of it downloading.

You need nothing from Sessions 1–2 to do it.

## Part 0 — Install the tools

One-time steps, in a **Terminal** (macOS/Linux) or **Anaconda Prompt** (Windows).

### Step 1 — An editor
Install [VS Code](https://code.visualstudio.com/), then from its Extensions panel install
**Python** and **Jupyter**, both by Microsoft.

### Step 2 — Python, via Miniconda
Download and run the installer for your platform from
<https://docs.conda.io/en/latest/miniconda.html>.

On **Windows**, tick *"Add Miniconda3 to my PATH"*. Almost every `command not found`
later traces back to that box.

### Step 3 — Create the course environment

```bash
conda create -n econ282e python=3.11 -y
conda activate econ282e
pip install numpy scipy pandas matplotlib jupyter torch tqdm statsmodels
```

`torch` here is the CPU build. **No GPU is needed for this course** until Session 7, and
even then a laptop is enough for the labs.

### Step 4 — Open this notebook and pick the kernel
In VS Code: **File → Open Folder** → the course `labs` folder → open this notebook →
click **Select Kernel** (top right) → **Python Environments** → `econ282e`.

Then run the next cell with **Shift+Enter**.

In [ ]:
# Run me first: does this environment have what the course needs?
import sys, importlib

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)

ready = True
for label, module in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"),
                      ("matplotlib", "matplotlib"), ("torch", "torch")]:
    try:
        m = importlib.import_module(module)
        print(f"  OK   {label:12s} {getattr(m, '__version__', '')}")
    except Exception:
        print(f"  MISS {label:12s} (not installed)")
        ready = False

print("\nSetup looks good." if ready else
      "\nInstall the missing packages (Part 0, Step 3), then re-run this cell.")

`Interpreter:` above should contain `econ282e`. If it does not, you are running a
different Python from the one you installed the packages into — pick the kernel again.
That single mismatch is the most common problem in this course.

## Part 1 — Your first Python

Three things that will come up in every lab.

In [ ]:
# 1. Variables have types, and you do not declare them.
alpha = 0.36          # float
periods = 200         # int
name = "Riverside"    # str
converged = False     # bool

print(type(alpha), type(periods))
print(f"alpha = {alpha}, that is {alpha:.1%} of output")

In [ ]:
# 2. Computers do not store the real numbers.
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

# So never test floats for equality. Test a distance against a tolerance:
print(abs((0.1 + 0.2) - 0.3) < 1e-12)

In [ ]:
# 3. Assignment never copies.
a = [1, 2, 3]
b = a              # a second NAME for one list
b[0] = 99
print("a is now", a)

c = a.copy()       # an actual copy
c[0] = 1
print("a is still", a)

## Part 2 — Your first macro model

The Solow model, $k_{t+1} = s k_t^{\alpha} + (1-\delta)k_t$, with the steady state
$k^{*} = (s/\delta)^{1/(1-\alpha)}$. We simulate it and check the simulation against
the closed form — which is the habit the whole of Session 3 is about.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

alpha, s, delta, T = 0.33, 0.20, 0.05, 200

k = 0.1
path = [k]
for t in range(T):
    k = s * k**alpha + (1 - delta) * k
    path.append(k)

k_star = (s / delta) ** (1 / (1 - alpha))

plt.figure(figsize=(8, 4))
plt.plot(path, lw=2, label="simulated $k_t$")
plt.axhline(k_star, color="firebrick", ls="--", label=f"closed form $k^*={k_star:.3f}$")
plt.xlabel("period"); plt.ylabel("capital per worker")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"simulated k_T = {path[-1]:.6f}")
print(f"closed form   = {k_star:.6f}")
print(f"difference    = {abs(path[-1] - k_star):.2e}")

**Exercise 1.** Raise the saving rate from `0.20` to `0.30` and re-run. Does $k^*$ move in
the direction you expected? By how much, in percent?

**Exercise 2.** Set `T = 20` instead of `200`. The simulation no longer reaches $k^*$.
How would you *detect* that automatically, without looking at the plot?

(The answer to Exercise 2 is a convergence criterion, and Session 3 spends a long time on
exactly how much it does and does not tell you.)

In [ ]:
# Exercise 2, one answer: stop when successive values stop moving, and report BOTH
# the criterion and the true error -- because they are not the same number.
k, tol = 0.1, 1e-8
for t in range(10_000):
    k_next = s * k**alpha + (1 - delta) * k
    step = abs(k_next - k)
    k = k_next
    if step < tol:
        break

print(f"stopped after {t+1} periods")
print(f"last step        = {step:.3e}")
print(f"true error       = {abs(k - k_star):.3e}")

Notice that the last step and the true error are **not** the same size. Session 3 explains
exactly what the relationship between them is, and why assuming they are equal is the most
common way to report a wrong number.

## You are ready

Bring a working environment to Session 3. The next lab, **L03**, solves the optimal growth
model end to end.